# Processing pipeline

- [ ] Stage 1. Correct background
    - inputs:
        - `fov/`: folder containing all the field of views extracted from the experiments.
        - `background/`: folder containing a background image that will be used to correct the background in all the field of views.
        - `proc_metadata/`: folder containing csv files to track the process.

    - outputs:

        - `fov_corrected/`: folder containing the images after background corrections.
        - `background_correction_metadata.csv`: a file containing metadata about the background correction process.

## Imports

In [ ]:
# Built-in imports
import logging


# Third-party imports
from pathlib import Path
from omegaconf import OmegaConf

# Package imports
from acid.image_processing.load_background_function import load_background_function
from acid.utils.filesystem.filesystem import create_output_directories
from acid.utils.metadata.loading import load_metadata
from acid.utils.metadata.filtering import filter_metadata_by_splits


# ----- Temporal (delete once refactored)
import numpy as np
from pprint import pprint

## Configure logging

In [ ]:
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

## Load Config File

In [ ]:
config = OmegaConf.load("config.yaml")

In [ ]:
# print(OmegaConf.to_yaml(config))

# Workflow

## 1. Create output and secondary output directory

In [ ]:
output_path, secondary_output_path = create_output_directories(
    output_directory=config["output"]["directory"],
    secondary_output_directory=config["output"]["secondary"]["directory"],
    enable_secondary_output=config["output"]["secondary"]["enabled"],
)

## 2. Open the metadata dataframe

In [ ]:
metadata_df, metadata_file_name = load_metadata(config["metadata"])

## 3. Select train data

In [ ]:
filtered_dataframe = filter_metadata_by_splits(
    metadata_df, selected_splits="train", split_config=config.metadata.dataset_split
)

In [ ]:
filtered_dataframe.info()

## 4. Read background data

### 4.3 New code

In [ ]:
background_result, background_filenames = load_background_function(
    background_config=config.background_correction,
    metadata_df=metadata_df,
)

In [ ]:
background_result.shape

In [ ]:
background_filenames

## 5. Main loop


### 5.1 Config

In [ ]:
# =============================================================================
# INPUT FIELD-OF-VIEW CONFIGURATION
# =============================================================================

# field of view column name in the metadata dataframe
fov_column_name = "ome_tif_file_name"

# indicate the path to the directory storing the fields of view
fov_directory = "data/proc/fov"

# within well position column name in the metadata dataframe
# this is used to identify the position of the field of view within the well
# 49 fields of view were acquired per each well, in a 7x7 grid
gridpos_column_name = "scene_name"

# well column name in the metadata dataframe
well_column_name = "well"


# =============================================================================
# GENERAL PROCESSING METADATA
# =============================================================================

# the data type of the image corrected for illumination
output_dtype = np.float32

# what is expected to be and what will be used as null value
null_value = np.nan

# --- parameters for saving metadata within the saved fields of view after illumination correction ---
# illumination correction image - name of processing date in metadata - this is the name
# of the entry in the metadata dictionary to save within the processed image.
# The entry indicates the date when the background correction was done.
proc_img_meta_date_name = "processing_date_yymmdd"

# illumination correction image - data type in metadata - this is the entry to use for indicating
# the data type of the illumination corrected image in the metadata saved within the processed image
proc_img_meta_dtype_name = "dtype"

# illumination correction image - date format in metadata - this is the format to use for indicating
# the date when the background correction was done in the metadata saved within the processed image
processing_date_format = "%y%m%d"


# =============================================================================
# ILLUMINATION CORRECTION IMAGE METADATA ENTRIES
# =============================================================================

# illumination correction image - illumination correction method in metadata - this is the entry to use for indicating
# the method used for illumination correction in the metadata saved within the processed image
illum_corr_method_metadata_entry = "illumination_correction_method"

# illumination correction image - illumination correction offset in metadata - this is the entry to use for indicating
# the offset used for illumination correction in the metadata saved within the processed image
illum_corr_offset_metadata_entry = "illumination_correction_offset"

# illumination correction image - illumination correction rescaling in metadata - this is the entry to use for indicating
# in the metadata saved within the processed image, whether or not the background function is rescaled when doing illumination correction
illum_corr_rescale_metadata_entry = "illumination_correction_rescale_background"

# illumination correction image - illumination correction clipping in metadata - this is the entry to use for indicating
# in the metadata saved within the processed image, whether or not the illumination corrected image is clipped
illum_corr_clipping_metadata_entry = "illumination_correction_clipping"

# illumination correction image - illumination correction clipping min value in metadata - this is the entry to use for
# indicating in the metadata saved within the processed image, what was the min value used for clipping (if clipping was performed)
illum_corr_clip_min_value_metadata_entry = "illumination_correction_min_clip_value"

# illumination correction image - illumination correction clipping max value in metadata - this is the entry to use for
# indicating in the metadata saved within the processed image, what was the max value used for clipping (if clipping was performed)
illum_corr_clip_max_value_metadata_entry = "illumination_correction_max_clip_value"

# illumination correction image - illumination correction offset_background in metadata - this is the entry to use for
# indicating in the metadata saved within the processed image, if the offset is subtracted to the background function
illum_corr_offset_background_metadata_entry = (
    "illumination_correction_offset_background"
)


# =============================================================================
# ILLUMINATION CORRECTION METHOD PARAMETERS
# =============================================================================
# background function computation strategy - this is the strategy used in part4a notebook for
# calculating the background function. Possible options are: 1,2,3.
# 1 - calculate a  background function per channel using all the fields of view in the dataset which are not flagged
# 2 - calculate a background function per each channel and well using the fields of view in the train set which are not
# flagged and belong to the well
# 3 - calculate a background function per each channel and grid position using the fields of view in the train set which are
# not flagged and belong to the same position (aka grid position) across different wells
background_function_strategy = 1

# the method used for illumination correction - this can be either "subtraction" or "division"
# method="subtraction"
method = "division"

# the offset to subtract from the image before correcting for the illumination
offset = 400

# whether or not to rescale the background function before carrying out illumination correction.
# Three options are available:
# 1) None -> no rescaling is done.
# 2) "max" -> the background function is divided by the max value. The resulting, normalized background function has 1 as max value.
# 3) "minmax" -> the background function is min-max normalized (background - min) / (max - min + epsilon)
rescale_background = "max"

# whether or not to clip the image corrected for illumination. This is a boolean variable, it can be True or False.
# If True, the image after illumination correction is clipped in the min_clip_value, max_clip_value range.
# For example: when subtraction is use the result can contain negative values. It is possible to set them to 0
# by setting clip_corrected_image=True and min_clip_value=0 (irrespective of the max_clip_value)
clip_corrected_image = False

# the min value to which illumination corrected image is clipped to if clip_corrected_image is set to True
min_clip_value = 0

# the max value to which illumination corrected image is clipped to if clip_corrected_image is set to True
max_clip_value = None

# whether or not to offset the background function before carrying out illumination correction.
# Two options are available: True or False
offset_background = True


# =============================================================================
# BACKGROUND CORRECTION ARRAY / NUMERICAL PARAMETERS
# =============================================================================

# --- parameters for background illumination correction ---
# channel axis to be passed to background calculating functions in order to calculate background functions per channel
# this is the axis along which the channels are organized in the field of view arrays.
# For example, if the field of view arrays have shape (channels, height, width), the channel axis is 0.
channel_axis = 0

# the small constant to add to the background in oder to avoid zero numbers in the background function, which
# would lead to mathematical instability (e.g. division by zero)
epsilon = 1e-8

# the data type to use for the mathematical operations (subtraction and division) during illumination correction.
# NOTE: the same data type is used for both the image to correct and the background function
working_dtype = np.float32

# additional parameters to pass to the initial container of the illumation corrected image during the correction process
# ref to the function correct_background within the image_processing.correct_background.py script for further details.
zero_kwargs = None

# select if printing process information during background correction
verbose = True


# =============================================================================
# OUTPUT IMAGE FILE NAMING AND SAVING
# =============================================================================

# ome suffix - used to save ome.tif files
ome_suffix = ".ome.tif"

# --- parameters for file saving ---
# separator used for saved file names
save_file_name_separator = "_"

# illumination correction image name - savingword - this is the word to use in the field of view file name
# to indicate that the file is an illumination corrected field of view
fov_illumin_corrected_savingword = "bg"

# # indicate the path to the directory where outputs will be saved
# NOTE: it the directory does not exist, the pipeline will try to create it
output_directory = "data/proc/fov_proc"

# indicate whether to save the background function image with ImageJ compatible
save_imagej_compatible = True

# indicate the photometric interpretation to be used when saving the ome.tif files
# NOTE: at the moment, only 'minisblack' has been tested
photometric = "minisblack"


# =============================================================================
# METADATA DATAFRAME COLUMN NAMING
# =============================================================================

# --- parameters for metadata dataframe updating ---
# column name word separator - this is the separator to use for separating the different parts of
# the column names to be added to the metadata dataframe
column_name_separator = "_"

# illumination correction metadata dataframe - computation date column name - this is the name of the column
# to be added to the metadata dataframe to indicate the day when the illumination correction was computed.
illum_correct_df_date_clm_name = (
    f"illumination{column_name_separator}correction{column_name_separator}date"
)

# illumination correction metadata dataframe - corrected file column name - this is the name of the column
# to be added to the metadata dataframe to indicate the name used for saving the illumination corrected file.
illum_correct_df_file_name_clm_name = f"illumination{column_name_separator}correction{column_name_separator}file{column_name_separator}name"

# illumination correction metadata dataframe - method for illumination correction column name - this is the name
# of the column to be added to the metadata dataframe to indicate the method used for illumination correction.
illum_correct_df_method_clm_name = (
    f"illumination{column_name_separator}correction{column_name_separator}method"
)

# illumination correction metadata dataframe - offset for illumination correction column name - this is the name
# of the column to be added to the metadata dataframe to indicate the offset used for illumination correction.
illum_correct_df_offset_clm_name = (
    f"illumination{column_name_separator}correction{column_name_separator}offset"
)

# illumination correction metadata dataframe - rescale for illumination correction column name - this is the name
# of the column to be added to the metadata dataframe to indicate if the background function was rescaled when
# doing illumination correction and the method used
illum_correct_df_rescale_clm_name = f"illumination{column_name_separator}correction{column_name_separator}rescale{column_name_separator}background"

# illumination correction metadata dataframe - clipping for illumination correction column name - this is the name
# of the column to be added to the metadata dataframe to indicate if doing illumination correction the corrected image
# has been clipped.
illum_correct_df_clipping_clm_name = f"illumination{column_name_separator}correction{column_name_separator}clip{column_name_separator}output"

# illumination correction metadata dataframe - clipping min value for illumination correction column name - this is
# the name of the column to be added to the metadata dataframe to indicate what was the minimum value used
# for clipping the corrected image (if clipping was done).
illum_correct_df_clip_min_value_clm_name = f"illumination{column_name_separator}correction{column_name_separator}min{column_name_separator}clip{column_name_separator}value"

# illumination correction metadata dataframe - clipping max value for illumination correction column name - this is
# the name of the column to be added to the metadata dataframe to indicate what was the maximum value used
# for clipping the corrected image (if clipping was done).
illum_correct_df_clip_max_value_clm_name = f"illumination{column_name_separator}correction{column_name_separator}max{column_name_separator}clip{column_name_separator}value"

# illumination correction metadata dataframe - offset_background for illumination correction column name - this is
# the name of the column to be added to the metadata dataframe to indicate if doing illumination correction the
# background function has been offsetted.
illum_correct_df_offset_background_clm_name = f"illumination{column_name_separator}correction{column_name_separator}offset{column_name_separator}background"


# =============================================================================
# METADATA DATAFRAME FILE SAVING
# =============================================================================

# illumination correction metadata dataframe - computation date format - this is the format to be used for
# indicating the date when the illumination correction was computed. This is used for saving the date in the
# metadata dataframe
illum_correct_df_meta_date_format = "%y%m%d"

# indicate the path to the directory storing the metadata file - NOTE: this is expected to be the metadata_df saved
# from either part4a or part3 notebook
metadata_directory = "data/proc/proc_metadata"

# processing metadata dataframe name - date format
metadata_date_format = "%Y%m%d"

# project
project_name = "ACID"

# processing metadata dataframe name - savingword
metadata_savingword = "metadata"

# include indexes when saving pandas dataframes as csv files
save_csv_index = (
    False  # if False, the index will not be saved as a separate column in the csv file
)

# processing metadata dataframe name - file suffix
metadata_file_suffix = f"part{save_file_name_separator}4b.csv"

In [ ]:
import os
import tifffile
import datetime
from enum import IntEnum

import numpy as np
from tqdm.notebook import tqdm

from acid.image_processing.extract_metadata import extract_ometif_imagej_metadata
from acid.image_processing.make_imagej_metadata import imagej_compatible_metadata_dict
from acid.image_processing.correct_background import correct_background

from acid.utils.save_image import tifffile_save_ometiff


class BackgroundFunctionStrategy(IntEnum):
    DATASET = 1
    WELL = 2
    GRID_POSITION = 3


## 1. initialize lists to collect processing metadata for updating - this will be used to update the metadata df
illum_correct_date_collection = []
illum_correct_file_name_collection = []
illum_correct_method_collection = []
illum_correct_offset_collection = []
illum_correct_rescale_collection = []
illum_correct_clipping_collection = []
illum_correct_clip_min_value_collection = []
illum_correct_clip_max_value_collection = []
illum_correct_offset_background_collection = []

files_with_errors = []


# 2. iterate through the rows of the metadata dataframe
for row in tqdm(metadata_df.index, desc="Applying background correction"):
    logging.info("================================================")

    # ---------
    # 1. Open field of view
    # ---------
    field_of_view_file = metadata_df.loc[row, fov_column_name]
    logging.info(f"Working on file: {field_of_view_file}")

    # form full path to the field of view
    field_of_view_path = os.path.join(fov_directory, str(field_of_view_file))
    logging.debug(f"Field of view path: {field_of_view_path}")

    # try to open the field of view
    try:
        field_of_view = tifffile.imread(field_of_view_path)
    except:
        logging.warning(
            f"Can't open {field_of_view_file}, skipping illumination correction"
        )
        files_with_errors.append(field_of_view_file)

        # add null_value to the collection lists
        illum_correct_date_collection.append(null_value)
        illum_correct_file_name_collection.append(null_value)
        illum_correct_method_collection.append(null_value)
        illum_correct_offset_collection.append(null_value)
        illum_correct_rescale_collection.append(null_value)
        illum_correct_clipping_collection.append(null_value)
        illum_correct_clip_min_value_collection.append(null_value)
        illum_correct_clip_max_value_collection.append(null_value)
        illum_correct_offset_background_collection.append(null_value)

        continue

    # ---------
    # 2. Update image metadata
    # ---------
    # 2.1 Get field of view metadata
    field_of_view_metadata = extract_ometif_imagej_metadata(field_of_view_path)

    # 2.2 Update the image metadata data
    # 2.2.1 Create new keys for the field of view metadata dictionary
    processing_date_metadata_entry = next(
        iter(imagej_compatible_metadata_dict({proc_img_meta_date_name: "dummy"}))
    )
    dtype_metadata_entry = next(
        iter(imagej_compatible_metadata_dict({proc_img_meta_dtype_name: "dummy"}))
    )

    # 2.2.2 Populate the new keys with real data
    field_of_view_metadata[processing_date_metadata_entry] = (
        datetime.datetime.now().strftime(processing_date_format)
    )

    field_of_view_metadata[dtype_metadata_entry] = output_dtype

    # 2.2.3 Create a dictionary storing the information to add to the metadata
    field_of_view_illcorr_metadata = {
        illum_corr_method_metadata_entry: method,
        illum_corr_offset_metadata_entry: offset,
        illum_corr_rescale_metadata_entry: rescale_background,
        illum_corr_clipping_metadata_entry: clip_corrected_image,
        illum_corr_clip_min_value_metadata_entry: min_clip_value,
        illum_corr_clip_max_value_metadata_entry: max_clip_value,
        illum_corr_offset_background_metadata_entry: offset_background,
    }

    # 2.3.4 Convert the dictionary so that entries are distinguishable from default ones
    field_of_view_illcorr_metadata = imagej_compatible_metadata_dict(
        field_of_view_illcorr_metadata
    )

    # 2.3.5 Update the metadata dictionary by adding the new information on illumination correction
    for k in field_of_view_illcorr_metadata:
        field_of_view_metadata[k] = field_of_view_illcorr_metadata[k]

    # ---------
    # STRATEGY 1
    # ---------
    if background_function_strategy == BackgroundFunctionStrategy.DATASET:
        # get the illumination corrected image
        illum_correct_field_of_view = correct_background(
            field_of_view,
            # background=background_function,
            background=background_result,
            method=method,
            channel_axis=channel_axis,
            offset=offset,
            epsilon=epsilon,
            working_dtype=working_dtype,
            output_dtype=output_dtype,
            rescale_background=rescale_background,
            clip_corrected_image=clip_corrected_image,
            min_clip_value=min_clip_value,
            max_clip_value=max_clip_value,
            zero_kwargs=zero_kwargs,
            offset_background=offset_background,
            verbose=verbose,
        )

        # save the corrected image
        # form the saving name of the file
        save_file_name = f"{field_of_view_file.removesuffix(ome_suffix)}{save_file_name_separator}{fov_illumin_corrected_savingword}{ome_suffix}"

        tifffile_save_ometiff(
            os.path.join(output_directory, save_file_name),
            data=illum_correct_field_of_view,
            imagej=save_imagej_compatible,
            photometric=photometric,
            metadata=field_of_view_metadata,
        )

        # update the metadata collection lists
        # add null_value to the collection lists
        illum_correct_date_collection.append(
            datetime.datetime.now().strftime(illum_correct_df_meta_date_format)
        )
        illum_correct_file_name_collection.append(save_file_name)
        illum_correct_method_collection.append(method)
        illum_correct_offset_collection.append(offset)
        illum_correct_rescale_collection.append(rescale_background)
        illum_correct_clipping_collection.append(clip_corrected_image)
        illum_correct_clip_min_value_collection.append(min_clip_value)
        illum_correct_clip_max_value_collection.append(max_clip_value)
        illum_correct_offset_background_collection.append(offset_background)

        # print progress update
        print("illumation correction done. File saved. Metadata updated")

    # ---------
    # STRATEGY 2
    # ---------
    elif background_function_strategy == BackgroundFunctionStrategy.WELL:
        # get well
        fov_well = metadata_df.loc[row, well_column_name]

        # map the field of view with its background function
        # well_background_function = bg_funct_file_dict[fov_well]
        well_background_function = background_filenames[fov_well]

        # get the illumination corrected image
        illum_correct_field_of_view_well = correct_background(
            field_of_view,
            background=well_background_function,
            method=method,
            channel_axis=channel_axis,
            offset=offset,
            epsilon=epsilon,
            working_dtype=working_dtype,
            output_dtype=output_dtype,
            rescale_background=rescale_background,
            clip_corrected_image=clip_corrected_image,
            min_clip_value=min_clip_value,
            max_clip_value=max_clip_value,
            zero_kwargs=zero_kwargs,
            offset_background=offset_background,
            verbose=verbose,
        )

        # save the corrected image
        # form the saving name of the file
        save_file_name_well = f"{field_of_view_file.removesuffix(ome_suffix)}{save_file_name_separator}{fov_illumin_corrected_savingword}{ome_suffix}"

        tifffile_save_ometiff(
            os.path.join(output_directory, save_file_name_well),
            data=illum_correct_field_of_view_well,
            imagej=save_imagej_compatible,
            photometric=photometric,
            metadata=field_of_view_metadata,
        )

        # update the metadata collection lists
        # add null_value to the collection lists
        illum_correct_date_collection.append(
            datetime.datetime.now().strftime(illum_correct_df_meta_date_format)
        )
        illum_correct_file_name_collection.append(save_file_name_well)
        illum_correct_method_collection.append(method)
        illum_correct_offset_collection.append(offset)
        illum_correct_rescale_collection.append(rescale_background)
        illum_correct_clipping_collection.append(clip_corrected_image)
        illum_correct_clip_min_value_collection.append(min_clip_value)
        illum_correct_clip_max_value_collection.append(max_clip_value)
        illum_correct_offset_background_collection.append(offset_background)

        # print progress update
        print("illumation correction done. File saved. Metadata updated")

    # ---------
    # STRATEGY 3
    # ---------
    elif background_function_strategy == BackgroundFunctionStrategy.GRID_POSITION:
        # get grid-position
        fov_scene = metadata_df.loc[row, gridpos_column_name]

        # map the field of view with its background function
        # scene_background_function = bg_funct_file_dict[fov_scene]
        scene_background_function = background_filenames[fov_scene]

        # get the illumination corrected image
        illum_correct_field_of_view_scene = correct_background(
            field_of_view,
            background=scene_background_function,
            method=method,
            channel_axis=channel_axis,
            offset=offset,
            epsilon=epsilon,
            working_dtype=working_dtype,
            output_dtype=output_dtype,
            rescale_background=rescale_background,
            clip_corrected_image=clip_corrected_image,
            min_clip_value=min_clip_value,
            max_clip_value=max_clip_value,
            zero_kwargs=zero_kwargs,
            offset_background=offset_background,
            verbose=verbose,
        )

        # save the corrected image
        # form the saving name of the file
        save_file_name_scene = f"{field_of_view_file.removesuffix(ome_suffix)}{save_file_name_separator}{fov_illumin_corrected_savingword}{ome_suffix}"

        tifffile_save_ometiff(
            os.path.join(output_directory, save_file_name_scene),
            data=illum_correct_field_of_view_scene,
            imagej=save_imagej_compatible,
            photometric=photometric,
            metadata=field_of_view_metadata,
        )

        # update the metadata collection lists
        # add null_value to the collection lists
        illum_correct_date_collection.append(
            datetime.datetime.now().strftime(illum_correct_df_meta_date_format)
        )
        illum_correct_file_name_collection.append(save_file_name_scene)
        illum_correct_method_collection.append(method)
        illum_correct_offset_collection.append(offset)
        illum_correct_rescale_collection.append(rescale_background)
        illum_correct_clipping_collection.append(clip_corrected_image)
        illum_correct_clip_min_value_collection.append(min_clip_value)
        illum_correct_clip_max_value_collection.append(max_clip_value)
        illum_correct_offset_background_collection.append(offset_background)

        # print progress update
        print("illumation correction done. File saved. Metadata updated")

    else:
        raise ValueError("background_function_strategy must be 1, 2 or 3")

# print progress update
print("illumination correction finished")

# ---------
# UPDATE PROCESSING METADATA DATA FRAME
# ---------
metadata_df[illum_correct_df_date_clm_name] = illum_correct_date_collection
metadata_df[illum_correct_df_file_name_clm_name] = illum_correct_file_name_collection
metadata_df[illum_correct_df_method_clm_name] = illum_correct_method_collection
metadata_df[illum_correct_df_offset_clm_name] = illum_correct_offset_collection
metadata_df[illum_correct_df_rescale_clm_name] = illum_correct_rescale_collection
metadata_df[illum_correct_df_clipping_clm_name] = illum_correct_clipping_collection
metadata_df[illum_correct_df_clip_min_value_clm_name] = (
    illum_correct_clip_min_value_collection
)
metadata_df[illum_correct_df_clip_max_value_clm_name] = (
    illum_correct_clip_max_value_collection
)
metadata_df[illum_correct_df_offset_background_clm_name] = (
    illum_correct_offset_background_collection
)

# save the updated metadata dataframe as a csv file
metadata_df_name = f"{datetime.datetime.now().strftime(metadata_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{metadata_savingword}{save_file_name_separator}{metadata_file_suffix}"
metadata_df.to_csv(
    os.path.join(metadata_directory, metadata_df_name), index=save_csv_index
)

# print progress update
print("metadata saved")
print("finished")